# Module 2 — Crawl & Trích Xuất Văn Bản Số Hóa (PDF Text & Web HTML)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caoban123/Tool-Annotation-Sinonom/blob/main/Crawl/notebooks/01_Crawl_Text.ipynb)

> 🚀 **Chế độ Mặc Định Google Colab (GPU Khuyến Nghị)**
> - Notebook đã được tối ưu chạy trực tiếp 1-click trên Google Colab.
> - Khuyến nghị bật GPU: Vào menu **Runtime** → **Change runtime type** → Chọn **T4 GPU** để tăng tốc nhận dạng gấp 10-20 lần.
> - Toàn bộ dữ liệu sách cổ (660MB PDF) và mô hình AI sẽ được tự động tải về phiên làm việc.

Notebook này thực hiện trích xuất dữ liệu từ các tài liệu số hóa đã chuẩn hóa:
- **Tự động kéo dữ liệu 660MB:** Tự động tải toàn bộ kho sách cổ PDF gốc từ Google Drive vào Colab.
- **Tự động phân loại:** Soi xét PDF để nhận diện **PDF_TEXT** (trích xuất text layer trực tiếp) hay **PDF_SCAN** (ảnh scan cần OCR).
- **Tự động gán mã:** Gán mã đề tài `HVB_01`, mã tác phẩm, document ID, tính mã SHA-256 provenance theo hợp đồng chuẩn.

In [ ]:
# 1. Khởi tạo môi trường Colab & Tự động tải kho dữ liệu 660MB
import os
import sys
import subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    colab_repo = Path('/content/Tool-Annotation-Sinonom')
    if not colab_repo.exists() and not (Path('.').resolve() / 'Crawl').exists():
        print('Đang tải mã nguồn hệ thống về Google Colab...')
        try:
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/caoban123/Tool-Annotation-Sinonom.git', str(colab_repo)], check=True, capture_output=True)
            print('✓ Tải mã nguồn thành công!')
        except Exception as e:
            print(f'Thông báo: {e}')
    for candidate in [colab_repo, Path('/content'), Path('.')]:
        if (candidate / 'Crawl').exists():
            os.chdir(str(candidate))
            break
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf', 'pypdfium2', 'beautifulsoup4', 'pydantic', 'requests', 'gdown'], check=False)

REPO_ROOT = Path('.').resolve()
while REPO_ROOT.name and not (REPO_ROOT / 'Crawl').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Crawl.utils.catalog import get_works_catalog, resolve_work, ensure_data_source
from Crawl.utils.id_generator import generate_project_id, generate_document_id, compute_sha256, build_sidecar_metadata
from Crawl.extractors.html_extractor import extract_from_html
from Crawl.extractors.pdf_text_extractor import extract_from_pdf_text
from OCR.utils.pdf_detector import inspect_pdf

# Tự động kéo 660MB tài liệu PDF gốc từ link Google Drive
source_dir = ensure_data_source(base_dir=str(REPO_ROOT / 'data' / 'source_materials'))
print('✓ Hệ thống sẵn sàng làm việc!')

In [ ]:
# 2. Lựa chọn tác phẩm cần trích xuất văn bản
catalog = get_works_catalog()
print('=== DANH MỤC CÁC TÁC PHẨM CÓ SẴN ===')
for idx, (key, info) in enumerate(catalog.items(), 1):
    print(f"[{idx}] {key:15s} | {info['name']} ({info['author']})")

# Lựa chọn tác phẩm: HVNTDDC, LTHCLC_Q20_25, LTHCLC_Q46_49, VIET_SU_YEU, DVDDTB
SELECTED_WORK_KEY = 'LTHCLC_Q20_25'
LANGUAGE = 'viet'  # 'han' hoặc 'viet'
START_PAGE = 1    # Trang bắt đầu
END_PAGE = 20     # Trang kết thúc (đặt None để trích xuất toàn bộ sách)

work_info = resolve_work(SELECTED_WORK_KEY, base_dir=str(source_dir))
target_file = work_info['han_abs_path'] if LANGUAGE == 'han' else work_info['viet_abs_path']
print(f"\nĐang chọn: {work_info['name']} ({work_info['ma_tac_pham']}) — Ngôn ngữ: {LANGUAGE.upper()}")
print(f"Đường dẫn file: {target_file}")

In [ ]:
# 3. Tự động kiểm tra PDF Text vs PDF Scan
pdf_info = inspect_pdf(target_file, sample_pages=5) if (target_file and Path(target_file).exists()) else {'pdf_type': 'PDF_MISSING', 'total_pages': 0}

if not target_file or not Path(target_file).exists() or pdf_info.get('total_pages', 0) == 0:
    print(f'[LƯU Ý] Chưa thấy file PDF hoặc file rỗng. Chuyển sang file mẫu kiểm thử: tests/fixtures/sample_text.pdf')
    target_file = str(REPO_ROOT / 'tests' / 'fixtures' / 'sample_text.pdf')
    pdf_info = inspect_pdf(target_file, sample_pages=5)

print('=== KẾT QUẢ TỰ ĐỘNG PHÂN TÍCH PDF ===')
print(f"Tệp tin:           {Path(target_file).name}")
print(f"Loại tài liệu:     {pdf_info['pdf_type']}")
print(f"Tổng số trang:     {pdf_info['total_pages']}")
print(f"Ký tự mẫu/trang:   {pdf_info['avg_chars_per_page']}")
print(f"Khuyến nghị:       {pdf_info['recommended_engine']}")
print(f"Nhận định:         {pdf_info['reason']}")

if pdf_info['pdf_type'] == 'PDF_SCAN':
    print('\n[LƯU Ý] Đây là bản scan ảnh. Hãy mở notebook OCR (OCR/notebooks/01_OCR_Han.ipynb hoặc 02_OCR_Viet.ipynb) để nhận dạng.')

In [ ]:
# 4. Trích xuất văn bản thực tế & Tự động gán mã định danh
output_dir = REPO_ROOT / 'data' / 'crawled'
output_dir.mkdir(parents=True, exist_ok=True)

result = extract_from_pdf_text(target_file, start_page=START_PAGE, end_page=END_PAGE)
extracted_text = result['text']

# Tính mã SHA-256 provenance và sinh Document ID, Project ID
source_sha256 = compute_sha256(target_file)
doc_id = generate_document_id(work_info['ma_tac_pham'], LANGUAGE, source_sha256)
project_id = generate_project_id(work_info['ma_de_tai'], work_info['ma_tac_pham'])

lines = [l.strip() for l in extracted_text.splitlines() if l.strip()]
print(f'\n✓ Đã trích xuất thành công: {len(extracted_text)} ký tự ({len(lines)} đoạn/câu)')
print(f'Document ID: {doc_id}')
print(f'Project ID:  {project_id}')
print(f'SHA-256:     {source_sha256[:16]}...')

print('\n--- 5 ĐOẠN ĐẦU TIÊN ---')
for line in lines[:5]:
    print('>', line)

In [ ]:
# 5. Lưu kết quả ra file chuẩn hóa & Tải về máy tính cá nhân
import json

out_txt = output_dir / f"{LANGUAGE}.txt"
out_meta = output_dir / f"{LANGUAGE}_metadata.json"

with open(out_txt, 'w', encoding='utf-8', newline='\n') as f:
    f.write(extracted_text)

meta = build_sidecar_metadata(
    ma_de_tai=work_info['ma_de_tai'],
    ma_tac_pham=work_info['ma_tac_pham'],
    tac_pham_name=work_info['name'],
    language=LANGUAGE,
    source_file_path=target_file,
    total_segments=len(lines),
    total_characters=len(extracted_text),
    extractor_type='pdf_text_extractor',
    hardware='colab'
)

with open(out_meta, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'✓ Đã lưu file text:     {out_txt}')
print(f'✓ Đã lưu file metadata: {out_meta}')

# Tự động tải file text về máy tính để nạp vào Web Studio cục bộ
if 'google.colab' in sys.modules:
    from google.colab import files
    print('\nĐang chuẩn bị tải file về máy tính của bạn...')
    files.download(str(out_txt))